In [2]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import os 
import seaborn as sns

In [3]:
def data_corrector(symbol):
    path = '../updatedtop20datasets/'
    filepath = f'{path}{symbol}.parquet'
    if not os.path.exists(filepath):
        print(f"There is no such file for the symbol {symbol}")
        return None
    data = pd.read_parquet(filepath)
    data = data.loc[:, ~data.columns.str.contains('Unnamed')]
    data['date'] = pd.to_datetime(data['date'])
    if data.isna().sum().sum() > 0:
        print(f"Missing Values Detected for symbol {symbol}, Fixed using forward fill")
        data = data.ffill()
    if data.duplicated().sum() > 0:
        data = data.drop_duplicates()
        print(f"There are duplicate values for stock {symbol}. Fixed using duplication drop")
    data.to_parquet(filepath, index = False)
    return data



In [4]:
### Creating datasets for the top 20 companies 
watchlist = {
    "AAPL":  "Apple",
    "MSFT":  "Microsoft", 
    "NVDA":  "Nvidia",
    "GOOGL": "Alphabet (Google)",
    "AMD":  "AMD",
    "META":  "Meta (Facebook)",
    "TSLA":  "Tesla",
    "NFLX": "Netflix",
    "LLY":   "Eli Lilly",
    "AVGO":   "Broadcom",
    "MU":    "Micron Technology",
    "QCOM":   "Qualcomm",
    "UNH":   "UnitedHealth",
    "WMT":   "Walmart",
    "MA":    "Mastercard",
    "JNJ":   "Johnson & Johnson",
    "PG":    "Procter & Gamble",
    "HD":    "Home Depot",
    "ORCL":  "Oracle",
    "JPM": "JPMorgan Chase"
}



In [5]:
### Data preprocessing for all the files 
for key,value in watchlist.items():
    data_corrector(key)

### Problem statements: Like I create a parquet file in one notebook and try to update it in the other notebook why does not it update

In [6]:
data = data_corrector("AAPL")
data.head(5)

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,Symbol
0,2008-01-02 00:00:00+00:00,194.84,200.26,192.55,199.27,38542100,5.832436,5.994681,5.763886,5.965046,1079179879,0.0,1.0,AAPL
1,2008-01-03 00:00:00+00:00,194.93,197.39,192.69,195.41,30073800,5.835131,5.908769,5.768077,5.849499,842067242,0.0,1.0,AAPL
2,2008-01-04 00:00:00+00:00,180.05,193.00,178.89,191.45,51994000,5.389705,5.777357,5.354981,5.730959,1455833455,0.0,1.0,AAPL
3,2008-01-07 00:00:00+00:00,177.64,183.60,170.23,181.25,74006900,5.317563,5.495973,5.095749,5.425627,2072195272,0.0,1.0,AAPL
4,2008-01-08 00:00:00+00:00,171.25,182.46,170.80,180.14,54422000,5.126282,5.461847,5.112811,5.392399,1523817523,0.0,1.0,AAPL


---
### Features Explanation 
### 1. adjClose = 

---
### Creating the new features out of the given columns 


In [12]:
def new_features(symbol):
    path = '../updatedtop20datasets/'
    file_path = f'{path}{symbol}.parquet'
    data = pd.read_parquet(file_path)
    data['price_return'] = data['adjClose'].diff() / data['adjClose'].shift(1)
    data['price_return'] = data['price_return'].fillna(0)
    data['Volatility_7Days'] = data['adjClose'].rolling(window = 7).std().fillna(0)
    data['Volatility_30Days'] = data['adjClose'].rolling(window = 30).std().fillna(0)
    data['Price_Swing'] = (data['adjHigh'] - data['adjLow']) / data['adjClose']
    data['volume_zscore7Days'] = (data['adjVolume'] - data['adjVolume'].rolling(window=7).mean()) / data['adjVolume'].rolling(window=7).std()
    data['volume_zscore7Days'] = data['volume_zscore7Days'].fillna(0)
    data['volume_zscore30Days'] = (data['adjVolume'] - data['adjVolume'].rolling(window=30).mean()) / data['adjVolume'].rolling(window=30).std()
    data['volume_zscore30Days'] = data['volume_zscore30Days'].fillna(0)
    ### Calculating the MASignal
    data['7Day'] = data['adjClose'].rolling(window=7).mean()
    data['30Day'] = data['adjClose'].rolling(window=30).mean()
    data['MASignal'] = (data['7Day'] > data['30Day']).astype(int)
    data['year'] = data['date'].dt.year
    data = data.drop(columns = ['7Day', '30Day'])
    return data
    

### What is rolling Volatility ?!
### Rolling Volatility measures how much a stock's price has been (moving up and down ) over a fixed window of a time. 
### Here we have taken 7 days and 30 days which means how volatile the stock was in the given window of time

---
### Price swing! 
### Price swing also helps to identify how volatile was the stock during the given day 

---
### Volume z score 
### The volume z score helps calculate how suprising is today's activity in terms of volume or total stocks traded. Rolling window of 7 days and 30 days allows us to calculate how weird was the stock in the span of 7 or 30 days 

In [8]:
for key,value in watchlist.items():
    new_features(key)

In [9]:
data = new_features("AAPL")
data.head(10)

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,...,splitFactor,Symbol,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year
0,2008-01-02 00:00:00+00:00,194.84,200.26,192.55,199.27,38542100,5.832436,5.994681,5.763886,5.965046,...,1.0,AAPL,0.000000,0.000000,0.0,0.0,0.000000,0.0,0,2008
1,2008-01-03 00:00:00+00:00,194.93,197.39,192.69,195.41,30073800,5.835131,5.908769,5.768077,5.849499,...,1.0,AAPL,0.000462,0.000000,0.0,0.0,0.000000,0.0,0,2008
2,2008-01-04 00:00:00+00:00,180.05,193.00,178.89,191.45,51994000,5.389705,5.777357,5.354981,5.730959,...,1.0,AAPL,-0.076335,0.000000,0.0,0.0,0.000000,0.0,0,2008
3,2008-01-07 00:00:00+00:00,177.64,183.60,170.23,181.25,74006900,5.317563,5.495973,5.095749,5.425627,...,1.0,AAPL,-0.013385,0.000000,0.0,0.0,0.000000,0.0,0,2008
4,2008-01-08 00:00:00+00:00,171.25,182.46,170.80,180.14,54422000,5.126282,5.461847,5.112811,5.392399,...,1.0,AAPL,-0.035972,0.000000,0.0,0.0,0.000000,0.0,0,2008
5,2008-01-09 00:00:00+00:00,179.40,179.50,168.30,171.30,64781500,5.370248,5.373241,5.037975,5.127778,...,1.0,AAPL,0.047591,0.000000,0.0,0.0,0.000000,0.0,0,2008
6,2008-01-10 00:00:00+00:00,178.02,181.00,175.41,177.58,52963400,5.328938,5.418143,5.250809,5.315767,...,1.0,AAPL,-0.007692,0.271172,0.0,0.0,0.038211,0.0,0,2008
7,2008-01-11 00:00:00+00:00,172.69,177.85,170.00,176.00,44010200,5.169387,5.323849,5.088864,5.268471,...,1.0,AAPL,-0.029940,0.231091,0.0,0.0,-0.651221,0.0,0,2008
8,2008-01-14 00:00:00+00:00,178.78,179.42,175.17,177.52,39301800,5.351688,5.370847,5.243625,5.313971,...,1.0,AAPL,0.035266,0.103063,0.0,0.0,-1.287387,0.0,0,2008
9,2008-01-15 00:00:00+00:00,169.04,179.22,164.66,177.72,83688500,5.060127,5.364860,4.929013,5.319958,...,1.0,AAPL,-0.054480,0.124738,0.0,0.0,1.540341,0.0,0,2008


### What is rolling Volatility ?!
### Rolling Volatility measures how much a stock's price has been (moving up and down ) over a fixed window of a time. 
### Here we have taken 7 days and 30 days which means how volatile the stock was in the given window of time

---
### Price swing! 
### Price swing also helps to identify how volatile was the stock during the given day 

---
### Volume z score 
### The volume z score helps calculate how suprising is today's activity in terms of volume or total stocks traded. Rolling window of 7 days and 30 days allows us to calculate how weird was the stock in the span of 7 or 30 days 

In [11]:
for key,value in watchlist.items():
    new_features(key)